# CIFAR-10 with a bottleneck ResNet

A guided walk-through of the `resnet_cifar` package: look at the data, train a short run,
inspect the curves and check individual predictions. Run it from the repository root
(`jupyter lab notebooks/walkthrough.ipynb`).

In [ ]:
import sys; sys.path.insert(0, "..")   # make the package importable from /notebooks

import json, numpy as np, torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

from resnet_cifar import TrainConfig, build_model
from resnet_cifar.model import count_parameters
from resnet_cifar.config import CLASS_NAMES
from resnet_cifar.data import make_loaders, denormalize
from resnet_cifar.train import fit

## 1. Configuration
A short 5-epoch run keeps the notebook quick; raise `epochs` for a real result.

In [ ]:
cfg = TrainConfig(epochs=5, output_dir="../runs/notebook", data_dir="../data")
model = build_model(cfg)
print(f"{count_parameters(model):,} trainable parameters")

## 2. A peek at the (augmented) training data

In [ ]:
train_loader, val_loader, test_loader = make_loaders(cfg)
images, labels = next(iter(train_loader))
grid = denormalize(images[:16]).permute(0, 2, 3, 1).numpy()
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img, y in zip(axes.ravel(), grid, labels[:16]):
    ax.imshow(img); ax.set_title(CLASS_NAMES[y], fontsize=8); ax.axis("off")
plt.show()

## 3. Train

In [ ]:
history = fit(cfg)

## 4. Learning curves

In [ ]:
from resnet_cifar.plots import plot_history
plot_history(history, "../runs/notebook/curves.png")
display(Image("../runs/notebook/curves.png"))

## 5. Evaluate on the test set
The command line tool writes a full report and all figures to `runs/notebook/figures/`.

In [ ]:
!cd .. && python -m resnet_cifar.evaluate --run-dir runs/notebook --data-dir data

In [ ]:
display(Image("../runs/notebook/figures/confusion_matrix.png"))
display(Image("../runs/notebook/figures/predictions.png"))